# Conditional Diffusion for Rare Arrhythmia Augmentation
**Team I5 — AI × Medical | COSE362 Machine Learning Project**

Members: Ayaka, Amani, Luai & Imran

This notebook implements a class-conditional 1D diffusion model that generates synthetic ECG beats for rare arrhythmia classes from the MIT-BIH Arrhythmia Database, and tests whether augmenting the training set with these synthetic beats improves rare-arrhythmia detection.

**Full baseline comparison (this is the headline experiment):**
1. **Baseline** — no augmentation
2. **SMOTE** — classical feature-space interpolation
3. **GAN (WGAN-GP)** — adversarial generation
4. **Unconditional diffusion** — diffusion without class control
5. **Conditional diffusion (ours)** — class-conditional generation

**Runtime:** ~40-55 minutes on a free Colab T4 GPU. Set Runtime → Change runtime type → GPU.

## 1. Setup and Imports

In [ ]:
# Install dependencies
!pip install wfdb imbalanced-learn --quiet
print("Dependencies installed.")

In [ ]:
import os
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, recall_score
from sklearn.model_selection import train_test_split
from collections import Counter
import wfdb

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

## 2. Load MIT-BIH Arrhythmia Database

We download MIT-BIH directly from PhysioNet. This is the dataset DiffECG and the GAN paper both used. 48 recordings, sampled at 360 Hz, with beat-level annotations.

In [ ]:
DATA_DIR = "mitdb"
if not os.path.exists(DATA_DIR):
    print("Downloading MIT-BIH from PhysioNet (this takes 2-5 minutes)...")
    wfdb.dl_database('mitdb', dl_dir=DATA_DIR)
    print("Download complete.")
else:
    print("Dataset already downloaded.")

records = sorted(set(f.split('.')[0] for f in os.listdir(DATA_DIR) if f.endswith('.dat')))
print(f"Number of records: {len(records)}")

## 3. Extract Heartbeats

We extract a fixed window of 186 samples around each annotated R-peak (90 before, 96 after) — about 520ms at 360Hz, one full heartbeat. We map MIT-BIH's single-character symbols to descriptive class names and keep finer-grained labels to expose the long-tail problem.

In [ ]:
BEAT_LABELS = {
    'N': 'Normal', 'L': 'LBBB', 'R': 'RBBB', 'A': 'APB', 'V': 'PVC',
    'F': 'Fusion', '/': 'Paced', 'f': 'FusionPaced', 'j': 'NodalEscape',
    'E': 'VentEscape', 'a': 'AberratedAPB', 'J': 'NodalPrem',
}
WINDOW_BEFORE, WINDOW_AFTER = 90, 96
BEAT_LENGTH = WINDOW_BEFORE + WINDOW_AFTER  # 186

def extract_beats(record_name, data_dir):
    path = os.path.join(data_dir, record_name)
    signal, _ = wfdb.rdsamp(path)
    ann = wfdb.rdann(path, 'atr')
    sig = signal[:, 0]  # MLII lead
    beats, labels = [], []
    for i, symbol in enumerate(ann.symbol):
        if symbol not in BEAT_LABELS:
            continue
        r = ann.sample[i]
        if r - WINDOW_BEFORE < 0 or r + WINDOW_AFTER > len(sig):
            continue
        beats.append(sig[r-WINDOW_BEFORE : r+WINDOW_AFTER])
        labels.append(BEAT_LABELS[symbol])
    return beats, labels

all_beats, all_labels = [], []
print("Extracting beats...")
for i, rec in enumerate(records):
    b, l = extract_beats(rec, DATA_DIR)
    all_beats.extend(b); all_labels.extend(l)
    if (i+1) % 10 == 0:
        print(f"  {i+1}/{len(records)} records, {len(all_beats)} beats")

X = np.array(all_beats, dtype=np.float32)
y = np.array(all_labels)
print(f"\nTotal beats: {len(X)} | shape: {X.shape} | classes: {len(set(y))}")

## 4. Class Imbalance Analysis — The Core Problem

This figure is the centerpiece of the motivation. Normal beats dominate; several clinically meaningful arrhythmias have only a few hundred samples.

In [ ]:
counts = Counter(y)
classes_sorted = sorted(counts.items(), key=lambda x: -x[1])
total = len(y)
print(f"{'Class':<15}{'Count':>8}{'%':>8}")
print("-"*31)
for cls, cnt in classes_sorted:
    print(f"{cls:<15}{cnt:>8}{100*cnt/total:>7.2f}%")

fig, ax = plt.subplots(figsize=(12, 5))
names = [c[0] for c in classes_sorted]
vals = [c[1] for c in classes_sorted]
colors = ['#2ecc71' if v >= 1000 else '#e74c3c' for v in vals]
ax.bar(names, vals, color=colors)
ax.set_yscale('log')
ax.set_ylabel('Number of Beats (log scale)')
ax.set_title('MIT-BIH Class Distribution — Long-Tailed Imbalance\n(red = rare classes, our augmentation targets)')
plt.xticks(rotation=45); plt.tight_layout()
plt.savefig('class_imbalance.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"\nNormal beats: {100*counts['Normal']/total:.1f}% of all data.")

## 5. Preprocessing and Train/Test Split

Drop ultra-rare classes (<50 samples), z-score normalize each beat, stratified 80/20 split.

In [ ]:
MIN_SAMPLES = 50
keep = [c for c, n in counts.items() if n >= MIN_SAMPLES]
mask = np.isin(y, keep)
Xf, yf = X[mask], y[mask]

def norm_beat(b):
    return (b - b.mean()) / (b.std() + 1e-8)
Xn = np.array([norm_beat(b) for b in Xf], dtype=np.float32)

class_list = sorted(set(yf))
label_to_idx = {c: i for i, c in enumerate(class_list)}
idx_to_label = {i: c for c, i in label_to_idx.items()}
yi = np.array([label_to_idx[c] for c in yf])
NUM_CLASSES = len(class_list)

X_train, X_test, y_train, y_test = train_test_split(
    Xn, yi, test_size=0.2, random_state=SEED, stratify=yi)

train_counts = Counter(y_train)
RARE_CLASSES = [i for i in range(NUM_CLASSES) if train_counts[i] < 1000]
print(f"Classes: {NUM_CLASSES} | Train: {len(X_train)} | Test: {len(X_test)}")
print(f"\nRare classes (augmentation targets):")
for i in RARE_CLASSES:
    print(f"  {idx_to_label[i]:<15} {train_counts[i]:>6} samples")
TARGET_COUNT = 1000  # target size per rare class after augmentation

## 6. Shared Classifier and Evaluation Function

All five experiments use the *exact same* 1D CNN, trained the same way. This is critical — it means any performance difference comes purely from the augmentation strategy, not from a different classifier. This makes the comparison fair.

In [ ]:
class ECGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float().unsqueeze(1)
        self.y = torch.from_numpy(y).long()
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

class ECGClassifier(nn.Module):
    def __init__(self, num_classes, L=BEAT_LENGTH):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 32, 7, padding=3)
        self.conv2 = nn.Conv1d(32, 64, 5, padding=2)
        self.conv3 = nn.Conv1d(64, 128, 3, padding=1)
        self.bn1, self.bn2, self.bn3 = nn.BatchNorm1d(32), nn.BatchNorm1d(64), nn.BatchNorm1d(128)
        self.pool = nn.MaxPool1d(2)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(128 * (L // 8), 64)
        self.fc2 = nn.Linear(64, num_classes)
    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.dropout(F.relu(self.fc1(x.flatten(1))))
        return self.fc2(x)

def train_classifier(X_tr, y_tr, X_te, y_te, num_epochs=15, batch_size=128, label=""):
    tr = DataLoader(ECGDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True, drop_last=True)
    te = DataLoader(ECGDataset(X_te, y_te), batch_size=batch_size, shuffle=False)
    model = ECGClassifier(NUM_CLASSES).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    print(f"\n=== Training classifier [{label}] ===")
    for epoch in range(num_epochs):
        model.train(); tot = 0
        for xb, yb in tr:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward(); opt.step()
            tot += loss.item()
        if (epoch+1) % 5 == 0:
            print(f"  epoch {epoch+1}/{num_epochs} loss={tot/len(tr):.4f}")
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for xb, yb in te:
            preds.extend(model(xb.to(DEVICE)).argmax(1).cpu().numpy())
            labels.extend(yb.numpy())
    f1 = f1_score(labels, preds, average=None, zero_division=0)
    macro = f1_score(labels, preds, average='macro', zero_division=0)
    rec = recall_score(labels, preds, average=None, zero_division=0)
    return {'per_class_f1': f1, 'macro_f1': macro, 'per_class_recall': rec}

## 7. Experiment 1 — Baseline (No Augmentation)

Train on the raw imbalanced data. This is the "before" we try to beat.

In [ ]:
res_baseline = train_classifier(X_train, y_train, X_test, y_test, label="BASELINE")
print(f"\nBaseline macro F1: {res_baseline['macro_f1']:.4f}")

## 8. Experiment 2 — SMOTE Augmentation

SMOTE oversamples rare classes by linear interpolation in feature space. The baseline our method must beat to be justified.

In [ ]:
from imblearn.over_sampling import SMOTE
X_flat = X_train.reshape(len(X_train), -1)
strategy = {i: max(TARGET_COUNT, train_counts[i]) for i in RARE_CLASSES}
k = min(5, max(1, min(train_counts[i] for i in RARE_CLASSES) - 1))
X_sm, y_sm = SMOTE(sampling_strategy=strategy, k_neighbors=k, random_state=SEED).fit_resample(X_flat, y_train)
X_sm = X_sm.reshape(-1, BEAT_LENGTH).astype(np.float32)
print(f"After SMOTE: {len(X_sm)} beats")
res_smote = train_classifier(X_sm, y_sm, X_test, y_test, label="SMOTE")
print(f"\nSMOTE macro F1: {res_smote['macro_f1']:.4f}")

## 9. Experiment 3 — GAN Augmentation (WGAN-GP)

A 1D conditional Wasserstein GAN with gradient penalty. We use WGAN-GP rather than a vanilla GAN because it's substantially more stable — but as the literature notes, GANs still struggle with mode collapse on physiological signals. This is one of the three baselines from our proposal.

In [ ]:
LATENT_DIM = 64

class Generator(nn.Module):
    def __init__(self, num_classes, latent_dim=LATENT_DIM, L=BEAT_LENGTH):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, latent_dim)
        self.L = L
        self.init_len = L // 4
        self.fc = nn.Linear(latent_dim * 2, 128 * self.init_len)
        self.net = nn.Sequential(
            nn.BatchNorm1d(128), nn.Upsample(scale_factor=2),
            nn.Conv1d(128, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Upsample(scale_factor=2),
            nn.Conv1d(64, 32, 5, padding=2), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32, 1, 5, padding=2),
        )
    def forward(self, z, labels):
        x = torch.cat([z, self.label_emb(labels)], dim=1)
        x = self.fc(x).view(x.size(0), 128, self.init_len)
        x = self.net(x)
        if x.size(-1) != self.L:
            x = F.interpolate(x, size=self.L)
        return x

class Critic(nn.Module):
    def __init__(self, num_classes, L=BEAT_LENGTH):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, L)
        self.net = nn.Sequential(
            nn.Conv1d(2, 32, 5, stride=2, padding=2), nn.LeakyReLU(0.2),
            nn.Conv1d(32, 64, 5, stride=2, padding=2), nn.LeakyReLU(0.2),
            nn.Conv1d(64, 128, 5, stride=2, padding=2), nn.LeakyReLU(0.2),
        )
        with torch.no_grad():
            flat = self.net(torch.zeros(1, 2, L)).flatten(1).size(1)
        self.fc = nn.Linear(flat, 1)
    def forward(self, x, labels):
        x = torch.cat([x, self.label_emb(labels).unsqueeze(1)], dim=1)
        return self.fc(self.net(x).flatten(1))

def gradient_penalty(critic, real, fake, labels):
    B = real.size(0)
    eps = torch.rand(B, 1, 1, device=DEVICE)
    interp = (eps * real + (1 - eps) * fake).requires_grad_(True)
    score = critic(interp, labels)
    grad = torch.autograd.grad(score, interp, torch.ones_like(score),
                               create_graph=True, retain_graph=True)[0].view(B, -1)
    return ((grad.norm(2, dim=1) - 1) ** 2).mean()

# Train WGAN-GP on rare classes
rare_mask = np.isin(y_train, RARE_CLASSES)
X_gan, y_gan = X_train[rare_mask], y_train[rare_mask]
gan_loader = DataLoader(ECGDataset(X_gan, y_gan), batch_size=128, shuffle=True, drop_last=True)
G = Generator(NUM_CLASSES).to(DEVICE)
D = Critic(NUM_CLASSES).to(DEVICE)
optG = torch.optim.Adam(G.parameters(), lr=1e-4, betas=(0.0, 0.9))
optD = torch.optim.Adam(D.parameters(), lr=1e-4, betas=(0.0, 0.9))
GAN_EPOCHS, N_CRITIC, LAMBDA = 40, 3, 10

print(f"=== Training WGAN-GP ({GAN_EPOCHS} epochs) ===")
for epoch in range(GAN_EPOCHS):
    for xb, yb in gan_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        B = xb.size(0)
        for _ in range(N_CRITIC):
            z = torch.randn(B, LATENT_DIM, device=DEVICE)
            fake = G(z, yb).detach()
            optD.zero_grad()
            lossD = -(D(xb, yb).mean() - D(fake, yb).mean()) + LAMBDA * gradient_penalty(D, xb, fake, yb)
            lossD.backward(); optD.step()
        z = torch.randn(B, LATENT_DIM, device=DEVICE)
        optG.zero_grad()
        lossG = -D(G(z, yb), yb).mean()
        lossG.backward(); optG.step()
    if (epoch+1) % 10 == 0:
        print(f"  epoch {epoch+1}/{GAN_EPOCHS} D={lossD.item():.3f} G={lossG.item():.3f}")

@torch.no_grad()
def gan_generate(class_idx, n):
    G.eval()
    z = torch.randn(n, LATENT_DIM, device=DEVICE)
    labels = torch.full((n,), class_idx, dtype=torch.long, device=DEVICE)
    return G(z, labels).cpu().numpy().squeeze(1)

# Build GAN-augmented set
gan_syn_X, gan_syn_y = [], []
for i in RARE_CLASSES:
    need = max(0, TARGET_COUNT - train_counts[i])
    if need > 0:
        gan_syn_X.append(gan_generate(i, need))
        gan_syn_y.append(np.full(need, i))
gan_syn_X = np.concatenate(gan_syn_X).astype(np.float32)
gan_syn_y = np.concatenate(gan_syn_y)
X_gan_aug = np.concatenate([X_train, gan_syn_X])
y_gan_aug = np.concatenate([y_train, gan_syn_y])
print(f"\nGAN-augmented set: {len(X_gan_aug)} beats (+{len(gan_syn_X)} synthetic)")
res_gan = train_classifier(X_gan_aug, y_gan_aug, X_test, y_test, label="GAN-AUGMENTED")
print(f"\nGAN macro F1: {res_gan['macro_f1']:.4f}")

## 10. Diffusion Model Components (shared by both diffusion experiments)

We define the U-Net building blocks and the DDPM noise schedule here. Both the unconditional baseline (Experiment 4) and our conditional method (Experiment 5) reuse these — the *only* difference between them is whether the U-Net receives the class label.

In [ ]:
T = 200
betas = torch.linspace(1e-4, 0.02, T).to(DEVICE)
alphas = 1 - betas
alpha_bars = torch.cumprod(alphas, dim=0)
sqrt_ab = torch.sqrt(alpha_bars)
sqrt_omab = torch.sqrt(1 - alpha_bars)

def q_sample(x0, t, noise):
    return sqrt_ab[t].view(-1,1,1) * x0 + sqrt_omab[t].view(-1,1,1) * noise

class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__(); self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        emb = math.log(10000) / (half - 1)
        emb = torch.exp(torch.arange(half, device=t.device) * -emb)
        emb = t[:, None].float() * emb[None, :]
        return torch.cat([emb.sin(), emb.cos()], dim=-1)

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, cond_dim):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv1d(out_ch, out_ch, 3, padding=1)
        self.norm1, self.norm2 = nn.GroupNorm(8, out_ch), nn.GroupNorm(8, out_ch)
        self.cond_proj = nn.Linear(cond_dim, out_ch)
        self.skip = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, cond):
        h = F.silu(self.norm1(self.conv1(x)))
        h = h + self.cond_proj(cond).unsqueeze(-1)
        h = F.silu(self.norm2(self.conv2(h)))
        return h + self.skip(x)

class UNet1D(nn.Module):
    """
    1D U-Net for ECG diffusion. If `conditional=True`, a class embedding is added
    to the time embedding. Setting conditional=False gives the unconditional baseline.
    This single flag is the ONLY difference between Experiments 4 and 5.
    """
    def __init__(self, num_classes, conditional=True, base_ch=64, cond_dim=128):
        super().__init__()
        self.conditional = conditional
        self.time_emb = SinusoidalTimeEmbedding(cond_dim)
        self.time_mlp = nn.Sequential(nn.Linear(cond_dim, cond_dim), nn.SiLU(), nn.Linear(cond_dim, cond_dim))
        if conditional:
            self.class_emb = nn.Embedding(num_classes, cond_dim)
        self.down1 = ConvBlock(1, base_ch, cond_dim)
        self.down2 = ConvBlock(base_ch, base_ch*2, cond_dim)
        self.down3 = ConvBlock(base_ch*2, base_ch*4, cond_dim)
        self.mid = ConvBlock(base_ch*4, base_ch*4, cond_dim)
        self.up3 = ConvBlock(base_ch*4+base_ch*4, base_ch*2, cond_dim)
        self.up2 = ConvBlock(base_ch*2+base_ch*2, base_ch, cond_dim)
        self.up1 = ConvBlock(base_ch+base_ch, base_ch, cond_dim)
        self.out_conv = nn.Conv1d(base_ch, 1, 1)
        self.pool = nn.AvgPool1d(2)
        self.up = nn.Upsample(scale_factor=2, mode='nearest')
    def forward(self, x, t, c=None):
        cond = self.time_mlp(self.time_emb(t))
        if self.conditional:
            cond = cond + self.class_emb(c)
        h1 = self.down1(x, cond)
        h2 = self.down2(self.pool(h1), cond)
        h3 = self.down3(self.pool(h2), cond)
        h = self.mid(self.pool(h3), cond)
        h = self.up(h)
        if h.shape[-1] != h3.shape[-1]: h = F.interpolate(h, size=h3.shape[-1])
        h = self.up3(torch.cat([h, h3], 1), cond)
        h = self.up(h)
        if h.shape[-1] != h2.shape[-1]: h = F.interpolate(h, size=h2.shape[-1])
        h = self.up2(torch.cat([h, h2], 1), cond)
        h = self.up(h)
        if h.shape[-1] != h1.shape[-1]: h = F.interpolate(h, size=h1.shape[-1])
        h = self.up1(torch.cat([h, h1], 1), cond)
        return self.out_conv(h)

print(f"Diffusion configured: T={T} steps")

## 11. Experiment 4 — Unconditional Diffusion

This diffusion model has the same architecture as our method but **no class conditioning**. Because it can't be told which class to generate, we train it only on the pooled rare-class beats, so it generates "rare-ish" beats — but it can't target a *specific* rare arrhythmia. This isolates exactly what the conditioning buys us. It's the third baseline from our proposal.

In [ ]:
def train_diffusion(model, X_data, y_data, epochs, conditional, label):
    loader = DataLoader(ECGDataset(X_data, y_data), batch_size=128, shuffle=True, drop_last=True)
    opt = torch.optim.Adam(model.parameters(), lr=2e-4)
    print(f"=== Training {label} ({epochs} epochs) ===")
    history = []
    for epoch in range(epochs):
        model.train(); tot = 0; n = 0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            t = torch.randint(0, T, (xb.size(0),), device=DEVICE)
            noise = torch.randn_like(xb)
            xt = q_sample(xb, t, noise)
            pred = model(xt, t, yb) if conditional else model(xt, t)
            opt.zero_grad()
            loss = F.mse_loss(pred, noise)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += loss.item(); n += 1
        history.append(tot/n)
        if (epoch+1) % 5 == 0:
            print(f"  epoch {epoch+1}/{epochs} loss={tot/n:.4f}")
    return history

@torch.no_grad()
def diffusion_sample(model, n, conditional, class_idx=None):
    model.eval()
    x = torch.randn(n, 1, BEAT_LENGTH, device=DEVICE)
    c = torch.full((n,), class_idx, dtype=torch.long, device=DEVICE) if conditional else None
    for t in reversed(range(T)):
        tb = torch.full((n,), t, dtype=torch.long, device=DEVICE)
        pred = model(x, tb, c) if conditional else model(x, tb)
        a, ab, b = alphas[t], alpha_bars[t], betas[t]
        mean = (1/torch.sqrt(a)) * (x - (b/torch.sqrt(1-ab)) * pred)
        x = mean + (torch.sqrt(b)*torch.randn_like(x) if t > 0 else 0)
    return x.cpu().numpy().squeeze(1)

# Train unconditional diffusion on rare-class beats only
DIFFUSION_EPOCHS = 30
uncond_diff = UNet1D(NUM_CLASSES, conditional=False).to(DEVICE)
dummy_y = np.zeros(len(X_gan), dtype=np.int64)  # labels unused
_ = train_diffusion(uncond_diff, X_gan, dummy_y, DIFFUSION_EPOCHS, conditional=False, label="UNCONDITIONAL diffusion")

# Generate: split evenly across rare classes (model can't target, so we assign labels for the augmented set proportionally)
uncond_syn_X, uncond_syn_y = [], []
for i in RARE_CLASSES:
    need = max(0, TARGET_COUNT - train_counts[i])
    if need > 0:
        uncond_syn_X.append(diffusion_sample(uncond_diff, need, conditional=False))
        uncond_syn_y.append(np.full(need, i))  # label assigned, but generation was NOT class-targeted
uncond_syn_X = np.concatenate(uncond_syn_X).astype(np.float32)
uncond_syn_y = np.concatenate(uncond_syn_y)
X_uncond_aug = np.concatenate([X_train, uncond_syn_X])
y_uncond_aug = np.concatenate([y_train, uncond_syn_y])
print(f"\nUnconditional-diffusion-augmented set: {len(X_uncond_aug)} beats")
res_uncond = train_classifier(X_uncond_aug, y_uncond_aug, X_test, y_test, label="UNCOND-DIFFUSION-AUGMENTED")
print(f"\nUnconditional diffusion macro F1: {res_uncond['macro_f1']:.4f}")

## 12. Experiment 5 — Conditional Diffusion (Our Method)

The same diffusion architecture, but now **class-conditional**. We can ask it to generate a *specific* rare arrhythmia. This is the only model that can do targeted rare-class generation. The class label is injected at every U-Net block.

In [ ]:
cond_diff = UNet1D(NUM_CLASSES, conditional=True).to(DEVICE)
print(f"Conditional U-Net parameters: {sum(p.numel() for p in cond_diff.parameters()):,}")
hist = train_diffusion(cond_diff, X_train, y_train, DIFFUSION_EPOCHS, conditional=True, label="CONDITIONAL diffusion")

plt.figure(figsize=(10, 4))
plt.plot(hist); plt.xlabel('Epoch'); plt.ylabel('DDPM Loss')
plt.title('Conditional Diffusion Training Loss'); plt.grid(alpha=0.3)
plt.savefig('diffusion_loss.png', dpi=120, bbox_inches='tight'); plt.show()

# Generate targeted rare-class beats
cond_syn_X, cond_syn_y = [], []
for i in RARE_CLASSES:
    need = max(0, TARGET_COUNT - train_counts[i])
    if need > 0:
        print(f"  generating {need} beats of '{idx_to_label[i]}'...")
        cond_syn_X.append(diffusion_sample(cond_diff, need, conditional=True, class_idx=i))
        cond_syn_y.append(np.full(need, i))
cond_syn_X = np.concatenate(cond_syn_X).astype(np.float32)
cond_syn_y = np.concatenate(cond_syn_y)
X_cond_aug = np.concatenate([X_train, cond_syn_X])
y_cond_aug = np.concatenate([y_train, cond_syn_y])
print(f"\nConditional-diffusion-augmented set: {len(X_cond_aug)} beats")
res_cond = train_classifier(X_cond_aug, y_cond_aug, X_test, y_test, label="CONDITIONAL-DIFFUSION (OURS)")
print(f"\nConditional diffusion macro F1: {res_cond['macro_f1']:.4f}")

## 13. Visual Inspection: Real vs. Synthetic (Our Method)

For each rare class, compare real beats to conditionally-generated synthetic ones. They should resemble the real beats in shape without being exact copies.

In [ ]:
fig, axes = plt.subplots(len(RARE_CLASSES), 2, figsize=(14, 3*len(RARE_CLASSES)))
if len(RARE_CLASSES) == 1: axes = axes.reshape(1, -1)
for row, i in enumerate(RARE_CLASSES):
    real = X_train[y_train == i][:5]
    fake = cond_syn_X[cond_syn_y == i][:5]
    for b in real: axes[row, 0].plot(b, alpha=0.7)
    axes[row, 0].set_title(f"REAL {idx_to_label[i]}"); axes[row, 0].grid(alpha=0.3)
    for b in fake: axes[row, 1].plot(b, alpha=0.7)
    axes[row, 1].set_title(f"SYNTHETIC {idx_to_label[i]} (conditional diffusion)"); axes[row, 1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig('real_vs_synthetic.png', dpi=120, bbox_inches='tight'); plt.show()

## 14. Final Comparison — All Five Settings

This is the headline result. We compare per-class F1 and rare-class macro F1 across all five experiments. The story we hope to see: **Conditional Diffusion > Unconditional Diffusion > GAN > SMOTE > Baseline** on rare-class detection.

In [ ]:
methods = ['Baseline', 'SMOTE', 'GAN', 'Uncond Diff', 'Cond Diff (Ours)']
results = [res_baseline, res_smote, res_gan, res_uncond, res_cond]

# Per-class F1 table
print("="*95)
print(f"{'Class':<14}" + "".join(f"{m:>16}" for m in methods))
print("-"*95)
for idx in range(NUM_CLASSES):
    rare = "*" if idx in RARE_CLASSES else " "
    row = f"{idx_to_label[idx]+rare:<14}"
    for r in results:
        row += f"{r['per_class_f1'][idx]:>16.4f}"
    print(row)
print("-"*95)
print(f"{'MACRO F1':<14}" + "".join(f"{r['macro_f1']:>16.4f}" for r in results))
print("="*95)
print("  (* = rare class)")

# Rare-class-only macro F1 — the KEY metric
def rare_macro(r): return np.mean([r['per_class_f1'][i] for i in RARE_CLASSES])
def rare_recall(r): return np.mean([r['per_class_recall'][i] for i in RARE_CLASSES])
rare_f1s = [rare_macro(r) for r in results]
rare_recs = [rare_recall(r) for r in results]

print(f"\n{'RARE-CLASS METRICS (the headline)':^60}")
print(f"{'Method':<20}{'Rare F1':>12}{'Rare Recall':>15}")
print("-"*47)
for m, f, rec in zip(methods, rare_f1s, rare_recs):
    print(f"{m:<20}{f:>12.4f}{rec:>15.4f}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
xp = np.arange(NUM_CLASSES)
w = 0.16
cols = ['#95a5a6', '#3498db', '#9b59b6', '#f39c12', '#e74c3c']
for k, (r, c, m) in enumerate(zip(results, cols, methods)):
    axes[0].bar(xp + (k-2)*w, r['per_class_f1'], w, label=m, color=c)
axes[0].set_xticks(xp); axes[0].set_xticklabels([idx_to_label[i] for i in range(NUM_CLASSES)], rotation=45, ha='right')
axes[0].set_ylabel('F1 Score'); axes[0].set_title('Per-Class F1 — All Methods'); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3, axis='y')

axes[1].bar(methods, rare_f1s, color=cols)
axes[1].set_ylabel('Rare-Class Macro F1'); axes[1].set_title('Rare-Class Detection (the metric that matters)')
for i, v in enumerate(rare_f1s):
    axes[1].text(i, v+0.005, f'{v:.3f}', ha='center', fontweight='bold')
axes[1].grid(alpha=0.3, axis='y'); plt.setp(axes[1].get_xticklabels(), rotation=20, ha='right')
plt.tight_layout(); plt.savefig('final_results.png', dpi=120, bbox_inches='tight'); plt.show()

## 15. Discussion

**What the five-way comparison tells us:**

The key comparison is the rare-class macro F1 ordering. Each step up the ladder isolates one factor:
- **Baseline → SMOTE**: does *any* augmentation help?
- **SMOTE → GAN**: does *learned* generation beat linear interpolation?
- **GAN → Unconditional Diffusion**: does diffusion's stability beat GAN's mode collapse?
- **Unconditional → Conditional Diffusion**: does *class targeting* — our core contribution — add value beyond generic diffusion?

That last comparison is the most important one for our novelty claim. If conditional beats unconditional, we've shown that the conditioning mechanism — not just "diffusion" in general — is what drives the improvement.

**Interpreting outcomes:**
- If `Conditional > Unconditional`, our specific contribution (class targeting) is validated.
- If `Conditional ≈ Unconditional`, the conditioning isn't helping at this scale — try more diffusion epochs or more samples per class.
- If diffusion methods underperform GAN/SMOTE, the diffusion models likely need more training (raise `DIFFUSION_EPOCHS` to 50+).

**Limitations to address in the final report:**
- Single train/test split → use 5-fold cross-validation
- Single-lead (MLII) input → extend to multi-lead
- No quality filtering of synthetic beats → add a confidence-based filter
- T=200 sampling steps for speed → T=1000 may improve sample quality

**Team task split for the final report:**
1. Cross-validation (Imran) 2. Multi-lead extension (Luai) 3. Quality filtering (Amani) 4. Final analysis & ablations (Ayaka)

In [ ]:
import json
out = {
    'methods': methods,
    'per_class_f1': {m: {idx_to_label[i]: float(r['per_class_f1'][i]) for i in range(NUM_CLASSES)}
                     for m, r in zip(methods, results)},
    'macro_f1': {m: float(r['macro_f1']) for m, r in zip(methods, results)},
    'rare_class_macro_f1': {m: float(f) for m, f in zip(methods, rare_f1s)},
    'rare_class_recall': {m: float(rec) for m, rec in zip(methods, rare_recs)},
    'rare_classes': [idx_to_label[i] for i in RARE_CLASSES],
}
with open('results.json', 'w') as f:
    json.dump(out, f, indent=2)
print("Saved results.json")
print(json.dumps(out['rare_class_macro_f1'], indent=2))